# Interactive Plotting

Interactively plot target_phot, comp_phot, and inspect_nights.

Author: Qiushi Chris Tian

Created: 2026-09-28

Updated: 2026-10-05

## Interactive Setting

Requires `ipyml`. Enable with either `%matplotlib ipympl` or `%matplotlib widget`.

In [ ]:
%matplotlib

In [ ]:
%matplotlib ipympl

## Imports and Style

In [ ]:
from pathlib import Path
from astropy import table
import numpy as np
import json
from datetime import datetime

from dep import plt, va

from optimize_rel_phot import RelativePhotometryEngine, plot_target, get_comps
from newport import TARGET_GAIA_DR3, get_excluded_comp, get_input_path

In [ ]:
plt.rcParams["font.family"] = "serif"
# plt.rcParams["font.serif"] = ["serif"]

In [ ]:
plt.rcParams["font.sans-serif"]

In [ ]:
plt.style.use('stix')

In [ ]:
plt.style.available

## `target_phot.py`

### Constants/Preamble

In [ ]:
TARGET = 'TOI-431'
TARGET_ID = TARGET_GAIA_DR3[TARGET]

OVERWRITE = False

TMIN = None
TMAX = None # datetime(2024, 4, 15)

FORCE_COMP = False

FORCE_COMP = True
FORCED_COMPS = [
    "2908637241099225600",
]
REF_COMPS = [
    "2908637172379753856",
    "2908637241099225600",
    "2908669156998939264",
    "2908681874399329280"
]

if FORCE_COMP:
    CRIT = np.nan
    OUTPUT_DIR = Path(
        f'data/tables/opt_comp_stars/{TARGET}/incomp-man-del_c1'
    )
else:
    CRIT = 0.999
    OUTPUT_DIR = Path(
        f'data/tables/opt_comp_stars/{TARGET}/'
        f'crit{str(CRIT).replace("0.", "")}_man_del'
    )
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
COMP_DIAG_DIR = OUTPUT_DIR / 'comp_diag'

### Main

In [ ]:
print(f'Newport photometry for target {TARGET} to {OUTPUT_DIR}')

full_table = table.Table.read(get_input_path(TARGET))

#### Optional: Print out comparison stars for comp diag

In [ ]:
COMP_DIAG_DIR.mkdir(parents=True, exist_ok=True)

comp_set = set()
for band in ['B', 'V', 'R', 'I']:
    json_path = OUTPUT_DIR / f"comps_{band}.json"
    comp_set.update(load_from_json(json_path, ['forced_comps', 'best_ensemble']))

for cid in comp_set:
    print(f"Gaia DR3 {cid}:")
    for band in ['B', 'V', 'R', 'I']:
        t = table.Table.read(COMP_DIAG_DIR / f"bin_diag_{cid}_{band}.fits")
        e = t.meta['COMPIDS']
        print(f"  {band}: {e}")

#### Optional: (Re-)Run Photometry

In [ ]:
bands = ['B', 'V', 'R', 'I']  # np.unique(full_table['band'])
all_binned = []
all_unbinned = []

for i, band in enumerate(bands):
    print(f"\n--- Band {band} ---")
    band_table = full_table[full_table['band'] == band]

    # Only 766452769615859712 has no V band data
    if len(band_table) == 0:
        print(f"No data for band {band}. Skipping.")
        continue
    
    # 1. Optimize (or use forced comps)
    if FORCE_COMP:
        best_ensemble = FORCED_COMPS
    else:
        best_ensemble, all_comps = get_comps(
            band_table, TARGET_ID, criterion=CRIT,
            exclude_ids=get_excluded_comp(TARGET, band)
        )
    
    # 2. Save results (includes binned/unbinned + metadata)
    engine = RelativePhotometryEngine(band_table, TARGET_ID)
    output_fn_base = OUTPUT_DIR / f"results_{band}"
    engine.save(best_ensemble, output_fn_base, sig_clip=3, overwrite=OVERWRITE)
    
    # Collect for stacking
    bin_t = table.Table.read(OUTPUT_DIR / f"bin_results_{band}.fits")
    unbin_t = table.Table.read(OUTPUT_DIR / f"unbin_results_{band}.fits")
    bin_t['band'] = band
    unbin_t['band'] = band
    all_binned.append(bin_t)
    all_unbinned.append(unbin_t)

    # 3. Save optimization summary
    summary = {
        "target": TARGET,
        "target_id": TARGET_ID,
        "band": band, 
    }

    if FORCE_COMP:
        summary.update({
            "forced_comps": FORCED_COMPS,
            "ref_comps": REF_COMPS,
        })
    else:
        summary.update({
            "best_ensemble": best_ensemble,
            "all_comps": all_comps,
        })
    
    output_json = OUTPUT_DIR / f"comps_{band}.json"
    with open(output_json, "w") as f:
        json.dump(summary, f, indent=4)
    print(f"Summary saved to {output_json}")
    print(best_ensemble)

# 4. Save consolidated tables
if all_binned:
    table.vstack(all_binned).write(OUTPUT_DIR / "bin_results_all.fits", overwrite=OVERWRITE)
    table.vstack(all_unbinned).write(OUTPUT_DIR / "unbin_results_all.fits", overwrite=OVERWRITE)
    print(f"\nConsolidated results saved to {OUTPUT_DIR}/[bin|unbin]_results_all.fits")

#### Optional: Plot

In [ ]:
plot_target(
    OUTPUT_DIR / "results",
    TARGET,
    # yrange=50,  # normally 12
    yrange='full',
    sharey=True,
    tmin=TMIN,
    tmax=TMAX,
    x_title=0.14,
    y_title=0.94,
    fig_width=6,
    fig_height_per_panel=1.3,
    sp_adj_top=0.92,
    # savefig_path=OUTPUT_DIR / f"photometric_monitoring_{TARGET}.pdf"
)

In [ ]:
plt.show()

In [ ]:
plt.gcf()

## `inspect_nights.py`

In [ ]:
import matplotlib.gridspec as gridspec
from matplotlib import dates as mdates, ticker
from astropy.time import Time
from zoneinfo import ZoneInfo
import newport
from inspect_nights import load_tables, find_flagged_nights, night_to_date_str

### Configs for `inspect_nights.py`

In [ ]:
EASTERN = ZoneInfo('US/Eastern')

# ── Configuration ──────────────────────────────────────────────
BANDS = ['B', 'V', 'R', 'I']

HIGH_STD = 0.1  # Flag night if intraday_std / flux > this in any band
HIGH_DEVI = 2  # Flag night if |flux - 1| > this many sigmas in any band

N_COLS = 4  # Number of columns in the bottom grid
MAX_ROWS = 10  # Maximum rows in the bottom grid
# (caps at N_COLS * MAX_ROWS nights)

SAVEFIG_NAME = 'anomalous_nights.pdf'

PLOT_UNBINNED = False
PLOT_AIRMASS = False  # Show airmass on twin y-axis in bottom panels

### Main

In [ ]:
bin_tables, unbin_tables = load_tables(OUTPUT_DIR, BANDS)
if not bin_tables:
    print("No tables found.")
    exit()

for band in BANDS:
    if TMIN:
        tmin_jd = Time(TMIN).jd
        bin_tables[band] = bin_tables[band][
            bin_tables[band]['jd'] >= tmin_jd
        ]
        unbin_tables[band] = unbin_tables[band][
            unbin_tables[band]['jd'] >= tmin_jd
        ]
    if TMAX:
        tmax_jd = Time(TMAX).jd
        bin_tables[band] = bin_tables[band][
            bin_tables[band]['jd'] <= tmax_jd
        ]
        unbin_tables[band] = unbin_tables[band][
            unbin_tables[band]['jd'] <= tmax_jd
        ]

# ── Find flagged nights ──
flagged_nights = find_flagged_nights(bin_tables, BANDS, HIGH_STD, HIGH_DEVI)
n_flagged = len(flagged_nights)
print(f'Flagged {n_flagged} nights:')
print([str(_) for _ in flagged_nights])

if n_flagged == 0:
    print("No anomalous nights found. Nothing to plot in bottom section.")

# Cap to grid capacity
n_flagged = min(n_flagged, N_COLS * MAX_ROWS)
flagged_nights = flagged_nights[:n_flagged]
n_bottom_rows = max(1, int(np.ceil(n_flagged / N_COLS))) if n_flagged > 0 else 0

# ── Figure layout ──
n_top_rows = len(BANDS)

fig = plt.figure(
    figsize=(10, 1.4 * n_top_rows + 1.6 * n_bottom_rows),
    # dpi=200
)

# Outer gridspec: top section vs bottom section
outer_gs = gridspec.GridSpec(
    2, 1, figure=fig,
    height_ratios=[n_top_rows * 1.4, n_bottom_rows * 1.6] if n_bottom_rows > 0 else [1, 0.001],
    hspace=0.12
)

# ── TOP SECTION: stacked BVRI ──
top_gs = gridspec.GridSpecFromSubplotSpec(
    n_top_rows, 1, subplot_spec=outer_gs[0], hspace=0
)
top_axs = []
for i in range(n_top_rows):
    share_x = top_axs[0] if i > 0 else None
    share_y = top_axs[0] if i > 0 else None
    ax = fig.add_subplot(
        top_gs[i], sharex=share_x,
        # sharey=share_y
    )
    top_axs.append(ax)
    if i < n_top_rows - 1:
        plt.setp(ax.get_xticklabels(), visible=False)

flagged_set = set(flagged_nights)

for i, band in enumerate(BANDS):
    ax = top_axs[i]
    if band not in bin_tables:
        continue
    bt = bin_tables[band]
    ut = unbin_tables[band]

    t_unbin = Time(ut['jd'], format='jd').to_datetime()
    t_bin = Time(bt['jd'], format='jd').to_datetime()

    # Catch ylim
    ax.errorbar(
        t_bin, (bt['flux'] - 1) * 1.2 + 1,
        fmt=newport.MARKERS[band], alpha=0, markeredgewidth=0, ms=7
    )
    ylim0, ylim1 = ax.get_ylim()

    # Unbinned (faint background)
    if PLOT_UNBINNED:
        ax.plot(
            t_unbin, ut['flux'], 'o', color='silver',
            ms=3, alpha=0.3, markeredgewidth=0
        )

    # Binned: split into flagged (full alpha) vs normal (dimmed)
    is_flagged = np.array([n in flagged_set for n in bt['night']])
    yerr = bt['intraday_std'] if 'intraday_std' in bt.colnames else None

    # Normal points (dimmed)
    if np.any(~is_flagged):
        ax.errorbar(
            np.array(t_bin)[~is_flagged], bt['flux'][~is_flagged],
            yerr=yerr[~is_flagged] if yerr is not None else None,
            fmt=newport.MARKERS[band], color=newport.COLORS[band],
            alpha=0.2, markeredgewidth=0, ms=7
        )

    # Flagged points (full alpha)
    if np.any(is_flagged):
        ax.errorbar(
            np.array(t_bin)[is_flagged], bt['flux'][is_flagged],
            yerr=yerr[is_flagged] if yerr is not None else None,
            fmt=newport.MARKERS[band], color=newport.COLORS[band],
            alpha=0.9, markeredgewidth=0, ms=7,
            # label=f'{band} band'
        )

    # Add MM-DD text labels for flagged nights (only on the bottom axis)
    if i == len(BANDS) - 1:
        labeled_nights = set()
        for row_idx in range(len(bt)):
            night = bt['night'][row_idx]
            if night in flagged_set and night not in labeled_nights:
                labeled_nights.add(night)
                t_pt = t_bin[row_idx]
                y_pt = bt['flux'][row_idx]
                ax.annotate(
                    night_to_date_str(night, mmdd=True),
                    (t_pt, y_pt),
                    textcoords='offset points',
                    xytext=(0, -18 if y_pt > 0.95 else 15),
                    fontsize=7, ha='center', va='top',
                    rotation=0, color='0.3'
                )

    ax.tick_params(axis='both', direction='in', labelsize=9)
    ax.xaxis.set_major_locator(mdates.AutoDateLocator(maxticks=8))
    ax.set_title(f'{band} band', loc='left', x=0.02, y=0.75)
    ax.grid(True, alpha=0.2)
    ax.set_ylim(ylim0, ylim1)

# # Top legend
# handles, labels = [], []
# for ax in top_axs:
#     h, l = ax.get_legend_handles_labels()
#     for hh, ll in zip(h, l):
#         if ll not in labels:
#             handles.append(hh)
#             labels.append(ll)
# fig.legend(
#     handles, labels, ncol=len(labels),
#     loc='upper center', bbox_to_anchor=(0.5, 1),
#     # fontsize=9
# )

# ── BOTTOM SECTION: per-night zoom-ins ──
if n_flagged > 0:
    bot_gs = gridspec.GridSpecFromSubplotSpec(
        n_bottom_rows, N_COLS, subplot_spec=outer_gs[1],
        hspace=0.4, wspace=0.3
    )

    bot_axs = []
    airmass_ref_ax = None
    for idx, night in enumerate(flagged_nights):
        # Row-first ordering: fill across rows first, then columns
        col = idx // n_bottom_rows
        row = idx % n_bottom_rows
        ax = fig.add_subplot(bot_gs[row, col])  # , sharey=share_y)
        bot_axs.append(ax)

        # Collect all data for this night across bands
        for band in BANDS:
            if band not in bin_tables:
                continue
            bt = bin_tables[band]
            ut = unbin_tables[band]

            # Unbinned data for this night (with error bars)
            u_mask = ut['night'] == night
            if np.any(u_mask):
                t_u = Time(ut['jd'][u_mask], format='jd').to_datetime(
                    timezone=EASTERN
                )
                ax.errorbar(
                    t_u, ut['flux'][u_mask],
                    yerr=ut['error'][u_mask] if 'error' in ut.colnames else None,
                    fmt=newport.MARKERS[band], color=newport.COLORS[band],
                    ms=4, alpha=0.4, markeredgewidth=0
                )

            # Binned data for this night (reference marker with intraday_std)
            b_mask = bt['night'] == night
            if np.any(b_mask):
                t_b = Time(bt['jd'][b_mask], format='jd').to_datetime(
                    timezone=EASTERN
                )
                ax.errorbar(
                    t_b, bt['flux'][b_mask],
                    yerr=bt['intraday_std'][b_mask] if 'intraday_std' in bt.colnames else None,
                    fmt=newport.MARKERS[band], color=newport.COLORS[band],
                    ms=7, alpha=0.8,
                    markeredgewidth=1, markeredgecolor=(0, 0, 0, 0.4)
                )

        title = night_to_date_str(night)
        ax.tick_params(axis='both', direction='in', labelsize=7)
        ax.grid(True, alpha=0.2)

        # X formatting: zoom to night, show hours in Eastern Time
        ax.xaxis.set_major_locator(ticker.MaxNLocator(nbins=4))
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
        # plt.setp(
        #     ax.get_xticklabels(),
        #     rotation=45, ha='right', fontsize=6
        # )

        # Collect airmass from unbinned data across all bands for this night
        airmass_t, airmass_v = [], []
        for band in BANDS:
            if band not in unbin_tables:
                continue
            ut = unbin_tables[band]
            u_mask = ut['night'] == night
            if np.any(u_mask):
                airmass_t.extend(
                    Time(ut['jd'][u_mask], format='jd').to_datetime(
                        timezone=EASTERN
                    )
                )
                airmass_v.extend(ut['airmass'][u_mask])

        if airmass_t:
            if PLOT_AIRMASS:
                # Sort by time for a connected line
                sort_idx = np.argsort(airmass_t)
                airmass_t = np.array(airmass_t)[sort_idx]
                airmass_v = np.array(airmass_v)[sort_idx]

                ax2 = ax.twinx()
                ax2.plot(
                    airmass_t, airmass_v, '-',
                    color='C1', lw=0.5, alpha=0.8
                )
                if col == N_COLS - 1:
                    ax2.set_ylabel(
                        'Airmass', x=1.03, fontsize=5, color='tan', rotation=270
                    )
                ax2.tick_params(
                    axis='y', labelsize=5, colors='tan',
                    direction='in'
                )
                # Share airmass y-axis across all bottom panels
                if airmass_ref_ax is None:
                    airmass_ref_ax = ax2
                else:
                    ax2.sharey(airmass_ref_ax)
            else:
                title += rf', $\langle \mathrm{{AM}} \rangle = {np.mean(airmass_v):.1f}$'

        ax.set_title(title, y=0.95, fontsize=8)

    # Hide unused bottom panels
    for idx in range(n_flagged, n_bottom_rows * N_COLS):
        col = idx // n_bottom_rows
        row = idx % n_bottom_rows
        ax = fig.add_subplot(bot_gs[row, col])
        ax.set_visible(False)

fig.supxlabel("Time of observation", y=0.07)
fig.supylabel("Normalized flux", x=0.07)
fig.subplots_adjust(top=0.96)

## Utility: Table Inspection

### Table(s) in `target_phot.py`

In [ ]:
np.unique(full_table['band'])

### Table(s) in `inspection_nights.py`

In [ ]:
t = unbin_tables['I']  # Naming used in inspection_nights.py

In [ ]:
t.sort('jd')

In [ ]:
t[t['night'] == 20231019]

### Original Photometry Table

In [ ]:
ot = table.Table.read('data/tables/list_runs/TOI-431/phot_list_run/phot_w_err_TOI-431.fits')

In [ ]:
ot.info

In [ ]:
ot.sort('jd')

In [ ]:
ot[(ot['band'] == 'I') & (ot['night'] == 20231019)][['time', 'jd', TARGET_ID, '2908637241099225600']]

In [ ]:
ot.colnames

In [ ]:
ot.info